The folder "data" contains around 2 millions tweets from the campaign, office, committee and party accounts of both houses of Congress, published between June 2017 and March 2020. There is one file per day containing that day’s tweets. I also have the "historical-users-filtered.json" dataset, which provides information on the political party of the accounts that wrote the collected tweets. 

I will combine these two datasets to label each tweet with the political party of its author and store all the data in one single file. I will alos filter the data to keep only the text of the tweet and the political party of its author. 

In [12]:
import pandas as pd
import json
import glob
import re
import os
from tqdm import tqdm

In [13]:
def clean_tweet(text):
    """
    Removes URLs from the tweet text (preserves hashtags, mentions, and punctuation.) """

    # Regex to match http, https, and www links
    url_pattern = r'https?://\S+|www\.\S+'
    text = re.sub(url_pattern, '', text)   #delete urls from the text
    
    # Strip leading/trailing whitespace
    return text.strip()

def load_and_map_users(filepath):
    """
    Creates a dictionary to map the user's ID and screen_name to their party.
    Republican = R, Democrat = D.
    Some users have "N/A" as a party, for now we will keep them under this labbel to count them 
    """
    party_mapping = {}
    
    with open(filepath, 'r', encoding='utf-8') as f:
        users_data = json.load(f)
        
    for user in users_data:
        party = user.get("party")

        for account in user.get("accounts"):
            account_id = account.get("id")
            party_mapping[str(account_id)] = party
                    
    return party_mapping

In [15]:
def build_dataset(data_dir, party_mapping):
    """
    Iterates through all JSON files, extracts tweets, and applies filters.
    """
    dataset = []
    
    # Find all .json files in the tweets folder and its subdirectories
    json_files = glob.glob(os.path.join(data_dir, '**', '*.json'), recursive=True)
    print(f"{len(json_files)} files found. Starting processing...")
    
    for file_path in tqdm(json_files, desc="Processing tweets"):
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                tweets = json.load(f)
                
            for tweet in tweets:
                user_id = str(tweet.get("user_id", ""))
                
                label = party_mapping.get(user_id) # Get the party for the user ID
                
                raw_text = tweet.get("text", "")
                cleaned_text = clean_tweet(raw_text)

                time = tweet.get("time", "")
                
                # Ignore tweets that became empty (e.g., tweet containing only a URL)
                if cleaned_text: 
                    dataset.append({
                        "text": cleaned_text,
                        "label": label,
                        "time" : time
                    })
        except Exception as e:
            print(f"Error reading file {file_path}: {e}")
            
    return pd.DataFrame(dataset)


In [16]:
# The script is in the same folder as data' and the JSON file
tweets_folder = "data"
users_file = "historical-users-filtered.json"
final_file = "congress_tweets_dataset.parquet"


print("Creating user mapping from historical-users-filtered.json")
mapping = load_and_map_users(users_file)

print("Extracting and cleaning tweets from the folder structure")
df_tweets = build_dataset(tweets_folder, mapping)

print("Saving the final merged dataset")
df_tweets.to_csv(final_file, index=False, encoding='utf-8')


# Display a small preview of the data
print(df_tweets.head())

Creating user mapping from historical-users-filtered.json
Extracting and cleaning tweets from the folder structure
2201 files found. Starting processing...


Processing tweets:   0%|          | 0/2201 [00:00<?, ?it/s]

Processing tweets: 100%|██████████| 2201/2201 [02:21<00:00, 15.54it/s]


Saving the final merged dataset
                                                text label  \
0  RT @CRN_Supplements Thank you @RepErikPaulsen ...     R   
1  Congrats to our Congressional Award Gold Medal...     R   
2  ICYMI: I chaired a hearing to explore expandin...     R   
3  The Affordable Care Act should be improved. Bu...     I   
4  #TaxReform will create a low tax rate just for...     R   

                        time  
0  2017-06-21T10:05:17-04:00  
1  2017-06-21T16:43:55-04:00  
2  2017-06-21T13:33:00-04:00  
3  2017-06-21T18:31:01-04:00  
4  2017-06-21T17:00:23-04:00  
